In [ ]:
!pip install sentencepiece -q
!pip install sacrebleu -q
!pip install nltk

In [ ]:
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.system("cp /kaggle/input/datasets/khimvng/checkpoints2/checkpoints/* /kaggle/working/checkpoints/")
print(os.listdir("/kaggle/working/checkpoints"))

In [ ]:
import torch

ckpt = torch.load("/kaggle/working/checkpoints/best_model.pt", map_location="cpu")
history = ckpt.get('history', None)

if history:
    for i, (tl, vl) in enumerate(zip(history['train_loss'], history['val_loss'])):
        print(f"Epoch {i+1}: Train Loss={tl:.4f} PPL={math.exp(tl):.2f} | Valid Loss={vl:.4f} PPL={math.exp(vl):.2f}")
else:
    print("Checkpoint không có history")
    print("Keys:", ckpt.keys())

In [ ]:
print(f"Epoch: {ckpt['epoch']}")
print(f"Train Loss: {ckpt['train_loss']:.4f}")
print(f"Valid Loss: {ckpt['valid_loss']:.4f}")
print(f"Config hidden_dim: {ckpt['config'].get('hidden_dim', '?')}")

In [ ]:
import sys
sys.path.append('/kaggle/input/datasets/khimvng/baseline20/baseline')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import time
import math
import os
# === IMPORT TRỰC TIẾP (không có baseline.) ===
from config_baseline import BASELINE_CONFIG
from data_utils import Vocabulary, build_bpe_vocabularies, load_bpe_vocabularies, TranslationDataset, collate_fn
from seq2seq_model import build_model, count_parameters
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
import os
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.system("cp -r /kaggle/input/datasets/khimvng/checkpoints2/checkpoints/* /kaggle/working/checkpoints/")
print("✓ Checkpoint loaded!")
print("Files:", os.listdir("/kaggle/working/checkpoints"))

In [ ]:
CONFIG = BASELINE_CONFIG
CONFIG["device"] = device
CONFIG["data_dir"] = "/kaggle/input/datasets/khimvng/dataset-tokenized2/split_dataset_tokenized"
CONFIG["checkpoint_dir"] = "/kaggle/working/checkpoints"
CONFIG["vocab_dir"] = "/kaggle/working/vocab"
CONFIG["batch_size"] =  64 # Tăng lên cho Kaggle GPU
CONFIG["bpe_vocab_size"] = 12000
CONFIG["epochs"] = 10
os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
os.makedirs(CONFIG["vocab_dir"], exist_ok=True)

In [ ]:
from datasets import load_dataset

full_dataset = load_dataset("ncduy/mt-en-vi")
dataset = {
    "train": full_dataset["train"].select(range(500000)),
    "validation": full_dataset["validation"],
    "test": full_dataset["test"]
}
print(f"\n Dataset info:")
print(f"   Train:      {len(dataset['train']):,} samples")
print(f"   Validation: {len(dataset['validation']):,} samples")
print(f"   Test:       {len(dataset['test']):,} samples")

In [ ]:
# Xem sample
print(" Sample data:")
print(f"\nEN: {dataset['train'][0]['en'][:200]}...")
print(f"VI: {dataset['train'][0]['vi'][:200]}...")

In [ ]:
# Build vocab
from data_utils import build_bpe_vocabularies
import shutil, os

if os.path.exists(CONFIG["vocab_dir"]):
    shutil.rmtree(CONFIG["vocab_dir"])
os.makedirs(CONFIG["vocab_dir"], exist_ok=True)

src_vocab, trg_vocab = build_bpe_vocabularies(
    dataset, vocab_size=CONFIG["bpe_vocab_size"], save_dir=CONFIG["vocab_dir"]
)
print(f"Source vocab: {len(src_vocab):,} | Target vocab: {len(trg_vocab):,}")

In [ ]:
# Create datasets
train_dataset = TranslationDataset(dataset["train"], src_vocab, trg_vocab, CONFIG["max_length"])
val_dataset = TranslationDataset(dataset["validation"], src_vocab, trg_vocab, CONFIG["max_length"])
test_dataset = TranslationDataset(dataset["test"], src_vocab, trg_vocab, CONFIG["max_length"])

# Create dataloaders
train_loader = DataLoader(
    train_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=True,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx),
    num_workers=0,
    pin_memory=True if device.type == 'cuda' else False
)

val_loader = DataLoader(
    val_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx),
    num_workers=0
)

test_loader = DataLoader(
    test_dataset,
    batch_size=CONFIG["batch_size"],
    shuffle=False,
    collate_fn=lambda b: collate_fn(b, src_vocab.pad_idx),
    num_workers=0
)

print(f" DataLoaders created!")
print(f"   Train batches: {len(train_loader)}")
print(f"   Val batches:   {len(val_loader)}")
print(f"   Test batches:  {len(test_loader)}")

In [ ]:
sample_batch = next(iter(train_loader))
print(" Sample batch shapes:")
print(f"   src: {sample_batch['src'].shape}")
print(f"   trg: {sample_batch['trg'].shape}")
print(f"   src_lengths: {sample_batch['src_lengths'].shape}")

In [ ]:
from seq2seq_model import build_model, count_parameters
# Build model
model = build_model(len(src_vocab), len(trg_vocab), CONFIG)

print(f" Model built!")
print(f"   Trainable parameters: {count_parameters(model):,}")
print(f"\n Model architecture:")
print(model)

In [ ]:
# Initialize weights
def init_weights(m):
    for name, param in m.named_parameters():
        if 'weight' in name:
            nn.init.normal_(param.data, mean=0, std=0.01)
        else:
            nn.init.constant_(param.data, 0)

model.apply(init_weights)
print(" Weights initialized!")

In [ ]:
from torch.cuda.amp import autocast, GradScaler
import random

def train_epoch(model, dataloader, optimizer, criterion, clip, device, teacher_forcing_ratio, scaler=None):
    """Train một epoch (FP16 Mixed Precision)"""
    model.train()
    epoch_loss = 0
    use_amp = scaler is not None
    
    progress_bar = tqdm(dataloader, desc="Training", leave=False)
    
    for batch in progress_bar:
        src = batch['src'].to(device)
        trg = batch['trg'].to(device)
        src_lengths = batch['src_lengths'].to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast(enabled=use_amp):
            output, coverage_loss = model(src, src_lengths, trg, teacher_forcing_ratio)
            output = output[:, 1:, :].contiguous().view(-1, output.size(-1))
            trg = trg[:, 1:].contiguous().view(-1)
            loss = criterion(output, trg)
        
        if use_amp:
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            scaler.step(optimizer)
            scaler.update()
        else:
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), clip)
            optimizer.step()
        
        epoch_loss += loss.item()
        progress_bar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    return epoch_loss / len(dataloader)


def evaluate(model, dataloader, criterion, device):
    """Evaluate model"""
    model.eval()
    epoch_loss = 0
    
    with torch.no_grad():
        for batch in tqdm(dataloader, desc="Evaluating", leave=False):
            src = batch['src'].to(device)
            trg = batch['trg'].to(device)
            src_lengths = batch['src_lengths'].to(device)
            
            with torch.cuda.amp.autocast(enabled=torch.cuda.is_available()):
                output, coverage_loss = model(src, src_lengths, trg, teacher_forcing_ratio=0)
                output = output[:, 1:, :].contiguous().view(-1, output.size(-1))
                trg = trg[:, 1:].contiguous().view(-1)
                loss = criterion(output, trg)
            epoch_loss += loss.item()
    
    return epoch_loss / len(dataloader)


def epoch_time(start_time, end_time):
    elapsed_time = end_time - start_time
    elapsed_mins = int(elapsed_time / 60)
    elapsed_secs = int(elapsed_time - (elapsed_mins * 60))
    return elapsed_mins, elapsed_secs

print(" Training functions defined! (FP16 enabled)")

In [ ]:
# Optimizer, Loss, Scheduler, FP16
optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"])
criterion = nn.CrossEntropyLoss(ignore_index=CONFIG["pad_idx"], label_smoothing=0.1)
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)
scaler = torch.cuda.amp.GradScaler() if torch.cuda.is_available() else None
print(f"✓ FP16: {'ENABLED' if scaler else 'DISABLED'}")
print(f"✓ Label Smoothing: 0.1")
print(f"✓ LR Scheduler: ReduceLROnPlateau")

# ===== CHECKPOINT RESUME SETUP =====
checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], 'last_checkpoint.pt')
start_epoch = 0

if os.path.exists(checkpoint_path):
    print(" Loading checkpoint để resume training...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_valid_loss = checkpoint.get('best_valid_loss', float('inf'))
    history = checkpoint.get('history', {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []})
    print(f" Resuming from epoch {start_epoch}")
else:
    print(" Starting fresh training")
    best_valid_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}

# ===== MAIN TRAINING LOOP =====
N_EPOCHS = CONFIG["epochs"]
print(f"\n Training {N_EPOCHS} epochs...")
print("=" * 60)

for epoch in range(start_epoch, N_EPOCHS):
    start_time = time.time()
    
    # Train (với FP16)
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion,
        CONFIG["grad_clip"], device, CONFIG["teacher_forcing_ratio"],
        scaler=scaler
    )
    
    # Evaluate
    valid_loss = evaluate(model, val_loader, criterion, device)
    
    # LR Scheduler step
    scheduler.step(valid_loss)
    
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(valid_loss)
    history['train_ppl'].append(math.exp(train_loss))
    history['val_ppl'].append(math.exp(valid_loss))
    
    # Save best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'valid_loss': valid_loss,
            'config': CONFIG
        }, os.path.join(CONFIG["checkpoint_dir"], 'best_model.pt'))
        star = ' ★ Best!'
    else:
        star = ''
    
    # Lưu checkpoint mỗi epoch (để resume khi hết 12h)
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_valid_loss': best_valid_loss,
        'history': history,
    }, checkpoint_path)
    
    current_lr = optimizer.param_groups[0]['lr']
    print(f"Epoch: {epoch+1:02}/{N_EPOCHS} | Time: {epoch_mins}m {epoch_secs}s | LR: {current_lr:.2e}")
    print(f"  Train Loss: {train_loss:.4f} | Train PPL: {math.exp(train_loss):7.2f}")
    print(f"  Valid Loss: {valid_loss:.4f} | Valid PPL: {math.exp(valid_loss):7.2f}{star}")

In [ ]:
# ===== CHECKPOINT RESUME SETUP =====
checkpoint_path = os.path.join(CONFIG["checkpoint_dir"], 'last_checkpoint.pt')
start_epoch = 0

# Load checkpoint nếu có (để resume training)
if os.path.exists(checkpoint_path):
    print(" Loading checkpoint để resume training...")
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint['model_state_dict'])
    optimizer.load_state_dict(checkpoint['optimizer_state_dict'])
    start_epoch = checkpoint['epoch'] + 1
    best_valid_loss = checkpoint.get('best_valid_loss', float('inf'))
    history = checkpoint.get('history', {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []})
    print(f"✓ Resuming from epoch {start_epoch}")
else:
    print(" Starting fresh training")
    best_valid_loss = float('inf')
    history = {'train_loss': [], 'val_loss': [], 'train_ppl': [], 'val_ppl': []}

# ===== MAIN TRAINING LOOP =====
N_EPOCHS = CONFIG["epochs"]

for epoch in range(start_epoch, N_EPOCHS):  # ← Bắt đầu từ start_epoch
    start_time = time.time()
    
    # Train
    train_loss = train_epoch(
        model, train_loader, optimizer, criterion,
        CONFIG["grad_clip"], device, CONFIG["teacher_forcing_ratio"]
    )
    
    # Evaluate
    valid_loss = evaluate(model, val_loader, criterion, device)
    
    end_time = time.time()
    epoch_mins, epoch_secs = epoch_time(start_time, end_time)
    
    # Save history
    history['train_loss'].append(train_loss)
    history['val_loss'].append(valid_loss)
    history['train_ppl'].append(math.exp(train_loss))
    history['val_ppl'].append(math.exp(valid_loss))
    
    # Save best model
    if valid_loss < best_valid_loss:
        best_valid_loss = valid_loss
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'valid_loss': valid_loss,
            'config': CONFIG
        }, os.path.join(CONFIG["checkpoint_dir"], 'best_model.pt'))
        star = ' Best!'
    else:
        star = ''
    
    # ===== LƯU CHECKPOINT MỖI EPOCH (QUAN TRỌNG!) =====
    torch.save({
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'best_valid_loss': best_valid_loss,
        'history': history,
    }, checkpoint_path)
    print(f" Checkpoint saved!")
    
    print(f"Epoch: {epoch+1:02}/{N_EPOCHS} | Time: {epoch_mins}m {epoch_secs}s")
    print(f"   Train Loss: {train_loss:.4f} | Train PPL: {math.exp(train_loss):7.2f}")
    print(f"   Valid Loss: {valid_loss:.4f} | Valid PPL: {math.exp(valid_loss):7.2f} {star}")
    print()

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history['train_loss'], label='Train Loss', marker='o')
axes[0].plot(history['val_loss'], label='Valid Loss', marker='o')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training & Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

axes[1].plot(history['train_ppl'], label='Train PPL', marker='o')
axes[1].plot(history['val_ppl'], label='Valid PPL', marker='o')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Perplexity')
axes[1].set_title('Training & Validation Perplexity')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('/kaggle/working/training_curves.png', dpi=150)  # ← Sửa ở đây
plt.show()

print(f"\n Best Validation Loss: {best_valid_loss:.4f}")
print(f" Best Validation PPL:  {math.exp(best_valid_loss):.2f}")

In [ ]:
def translate_sentence(model, sentence, src_vocab, trg_vocab, device, max_len=100):
    model.eval()
    
    with torch.no_grad():
        src_indices = src_vocab.encode(sentence)
        src_tensor = torch.LongTensor(src_indices).unsqueeze(0).to(device)
        src_lengths = torch.LongTensor([len(src_indices)]).to(device)
        
        # Translate (Beam Search)
        translations, attention_weights = model.beam_search_translate(
            src_tensor, src_lengths, max_len, beam_width=5
        )
        
        output_indices = translations[0].cpu().tolist()
        translated = trg_vocab.decode(output_indices)
        
    return translated, None

In [ ]:
# Load best model
checkpoint = torch.load(os.path.join(CONFIG["checkpoint_dir"], 'best_model.pt'))
model.load_state_dict(checkpoint['model_state_dict'])
print(f" Loaded best model from epoch {checkpoint['epoch'] + 1}")

# Test translations
test_sentences = [
    "The patient was diagnosed with diabetes.",
    "This study investigates the effects of exercise on health.",
    "The results show significant improvement."
]

print("\n" + "="*60)
print(" SAMPLE TRANSLATIONS")
print("="*60)

for sentence in test_sentences:
    translation, _ = translate_sentence(model, sentence, src_vocab, trg_vocab, device)
    print(f"\n EN: {sentence}")
    print(f" VI: {translation}")
    print("-"*60)

In [ ]:
# Test với mẫu từ test set
print("\n" + "="*60)
print(" TEST SET SAMPLES")
print("="*60)

for i in range(5):
    sample = test_dataset[i]
    src_text = sample['src_text']
    ref_text = sample['trg_text']
    
    pred_text, _ = translate_sentence(model, src_text, src_vocab, trg_vocab, device)
    
    print(f"\n[{i+1}]")
    print(f" Source (EN):    {src_text[:150]}...")
    print(f" Reference (VI): {ref_text[:150]}...")
    print(f" Predicted (VI): {pred_text[:150]}...")
    print("-"*60)

In [ ]:
from sacrebleu.metrics import BLEU
def calculate_bleu(model, dataloader, src_vocab, trg_vocab, device, num_samples=None):
    model.eval()
    
    predictions = []
    references = []
    
    bleu = BLEU()
    
    with torch.no_grad():
        for i, batch in enumerate(tqdm(dataloader, desc="Calculating BLEU")):
            if num_samples and i * dataloader.batch_size >= num_samples:
                break
                
            src = batch['src'].to(device)
            src_lengths = batch['src_lengths'].to(device)
            trg = batch['trg']
            
            # Beam search chỉ hỗ trợ batch_size=1, nên dịch từng câu
            for j in range(src.size(0)):
                actual_len = src_lengths[j].item()
                src_j = src[j, :actual_len].unsqueeze(0)        # [1, src_len]
                src_len_j = src_lengths[j].unsqueeze(0) # [1]
                
                translations, _ = model.beam_search_translate(src_j, src_len_j, beam_width=5)
                
                pred_indices = translations[0].cpu().tolist()
                ref_indices = trg[j].tolist()
                
                pred_text = trg_vocab.decode(pred_indices)
                ref_text = trg_vocab.decode(ref_indices)
                
                predictions.append(pred_text)
                references.append(ref_text)
    
    bleu_score = bleu.corpus_score(predictions, [references])
    
    return bleu_score, predictions, references

In [ ]:
# Calculate BLEU + METEOR + TER
from sacrebleu.metrics import TER
from nltk.translate.meteor_score import meteor_score

print(" Calculating metrics on test set...")
bleu_score, preds, refs = calculate_bleu(
    model, test_loader, src_vocab, trg_vocab, device, num_samples=500
)

# METEOR
meteor_scores = []
for pred, ref in zip(preds, refs):
    score = meteor_score([ref.split()], pred.split())
    meteor_scores.append(score)
avg_meteor = sum(meteor_scores) / len(meteor_scores) * 100

# TER
ter = TER()
ter_score = ter.corpus_score(preds, [refs])

print("\n" + "="*60)
print(" EVALUATION RESULTS")
print("="*60)
print(f"\n BLEU Score: {bleu_score.score:.2f}")
print(f"   Precisions: {[f'{p:.1f}' for p in bleu_score.precisions]}")
print(f"   BP: {bleu_score.bp:.4f}")
print(f"   Ratio: {bleu_score.sys_len / bleu_score.ref_len:.4f}")
print(f"\n METEOR Score: {avg_meteor:.2f}")
print(f"\n TER Score: {ter_score.score:.2f}")
print("="*60)

In [ ]:
# Save results
results = {
    'model': 'Seq2Seq + Bahdanau Attention (Bi-LSTM)',
    'params': count_parameters(model),
    'epochs': N_EPOCHS,
    'best_valid_loss': best_valid_loss,
    'best_valid_ppl': math.exp(best_valid_loss),
    'bleu_score': bleu_score.score,
    'config': {k: str(v) for k, v in CONFIG.items()},
    'history': history
}

import json
with open('/kaggle/working/results.json', 'w') as f:  # ← Sửa path
    json.dump(results, f, indent=2)

print(" Results saved to /kaggle/working/results.json")

# Summary
print("\n" + "="*60)
print(" TRAINING SUMMARY")
print("="*60)
print(f"   Model: Seq2Seq + Bahdanau Attention")
print(f"   Parameters: {count_parameters(model):,}")
print(f"   Best Valid Loss: {best_valid_loss:.4f}")
print(f"   Best Valid PPL: {math.exp(best_valid_loss):.2f}")
print(f"   BLEU Score: {bleu_score.score:.2f}")
print("="*60)
print("\n Baseline training completed!")
print(" Model saved to: /kaggle/working/checkpoints/best_model.pt")